In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"


# In[2]:


# %% [code]
# %% [code]
"""
Full GPT single-file implementation with ALL LayerNorm removed and fixed residual
scaling alpha = 1/sqrt(L) applied to each residual branch (attention and MLP).

Notes:
- No learnable alpha (no ReZero). Uses fixed scale = 1.0 / math.sqrt(config.n_layer).
- Removing LayerNorm means pretrained checkpoints (GPT-2 / HF) will not map 1:1.
  The from_pretrained helper will attempt to copy matching parameters and skip mismatches,
  but fine-tuning / reinitialization is recommended after loading.
"""

import math
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a single linear
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        # self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        self.flash = False
        if not self.flash:
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()  # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(
                q, k, v,
                attn_mask=None,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=True
            )
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v  # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)

        y = y.transpose(1, 2).contiguous().view(B, T, C)  # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


# class Block(nn.Module):
#     """
#     Transformer block WITHOUT LayerNorms.
#     Each residual branch is scaled by a fixed factor alpha = 1/sqrt(n_layer).
#     """

#     def __init__(self, config):
#         super().__init__()
#         # no LayerNorms anywhere
#         self.attn = CausalSelfAttention(config)
#         self.mlp = MLP(config)
#         # fixed scaling factor (not learnable)
#         self.scale = 1.0 / math.sqrt(config.n_layer)

#     def forward(self, x):
#         # Note: no ln_1 or ln_2; just raw residual plus scaled sublayer output
#         x = x + ( self.scale * self.attn(x) )
#         x = x + ( self.scale * self.mlp(x) )
#         return x


class Block(nn.Module):
    """
    Transformer block WITH Pre-LayerNorm.
    Each residual branch is scaled by a fixed factor alpha = 1/sqrt(n_layer).
    """

    def __init__(self, config):
        super().__init__()
        self.attn = CausalSelfAttention(config)
        self.mlp = MLP(config)

        # --- Added: Pre-LayerNorm for attention and MLP ---
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.ln_2 = nn.LayerNorm(config.n_embd)

        # fixed scaling factor (not learnable)
        #self.scale_1 = nn.Parameter( torch.tensor(0.0)) #1.0 / math.sqrt(config.n_layer))
        #self.scale_2 = nn.Parameter( torch.tensor(0.0)) 
        self.scale = nn.Parameter( torch.tensor(0.0)) 
    def forward(self, x):
        # Pre-LN attention
        x = x + self.scale * self.attn(self.ln_1(x))
        

        # Pre-LN MLP
        x = x +  self.scale *self.mlp(self.ln_2(x))
        

        return x


@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304  # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 6
    n_head: int = 8
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True  # True: bias in Linears, False: faster slightly


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        # transformer modules
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            # NOTE: removed ln_f (final LayerNorm) entirely
        ))
        # language model head
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # weight tying: share token embedding and lm_head weights
        # this keeps behavior similar to original GPT implementations
        self.transformer.wte.weight = self.lm_head.weight

        # init all weights
        self.apply(self._init_weights)

        # scaled init for residual projection weights (like GPT-2)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params() / 1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return number of parameters. If non_embedding=True subtract position embeddings
        to match many published counts.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # def forward(self, idx, targets=None):
    #     """
    #     idx: (b, t) long tensor of token indices
    #     targets: (b, t) long tensor for computing cross-entropy loss (optional)
    #     """
    #     device = idx.device
    #     b, t = idx.size()
    #     assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
    #     pos = torch.arange(0, t, dtype=torch.long, device=device)  # shape (t)

    #     # token + position embeddings
    #     tok_emb = self.transformer.wte(idx)  # (b, t, n_embd)
    #     pos_emb = self.transformer.wpe(pos)  # (t, n_embd)
    #     x = self.transformer.drop(tok_emb + pos_emb)

    #     # forward through transformer blocks (no LayerNorms)
    #     for block in self.transformer.h:
    #         x = block(x)

    #     # no final LayerNorm (ln_f) is applied

    #     if targets is not None:
    #         # full logits for loss computation
    #         logits = self.lm_head(x)  # (b, t, vocab)
    #         loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
    #     else:
    #         # during inference, only compute logits for last token to save compute
    #         logits = self.lm_head(x[:, [-1], :])  # (b, 1, vocab)
    #         loss = None

    #     return logits, loss
    def forward(self, idx, targets=None, return_hidden_states=False):
        """
        idx: (b, t) long tensor of token indices
        targets: optional (b, t)
        return_hidden_states:
            if True, returns hidden states:
            hidden_states[0] = token + position embedding output
            hidden_states[i] = output after transformer block i
        """
        device = idx.device
        b, t = idx.size()
    
        assert t <= self.config.block_size, (
            f"Cannot forward sequence of length {t}, "
            f"block size is only {self.config.block_size}"
        )
    
        pos = torch.arange(0, t, dtype=torch.long, device=device)
    
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
    
        hidden_states = None
        if return_hidden_states:
            hidden_states = [x]
    
        for block in self.transformer.h:
            x = block(x)
            if return_hidden_states:
                hidden_states.append(x)
    
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1
            )
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
    
        if return_hidden_states:
            return logits, loss, tuple(hidden_states)
    
        return logits, loss

    def crop_block_size(self, block_size):
        """
        Reduce block size (positional embeddings and attention mask) for memory savings.
        """
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                block.attn.bias = block.attn.bias[:, :, :block_size, :block_size]



    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # filter parameters that require gradients
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        # decay parameters are those with dim >= 2
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")

        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, eps=1e-4, **extra_args)
        print(f"using fused AdamW: {use_fused}")
        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        """ estimate model flops utilization (MFU) in units of A100 bfloat16 peak FLOPS """
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd // cfg.n_head, cfg.block_size
        flops_per_token = 6 * N + 12 * L * H * Q * T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        flops_achieved = flops_per_iter * (1.0 / dt)
        flops_promised = 312e12  # A100 bfloat16 peak flops ~312 TFLOPS
        mfu = flops_achieved / flops_promised
        return mfu

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        autoregressive generation: append tokens to idx max_new_tokens times
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [3]:
import os
import json
import math
import pickle
import random
import numpy as np
import torch
from tqdm import tqdm
from contextlib import nullcontext

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

CKPT_PATH = "./ckpt_best_bs_512_learnable_alpha.pt"   
# 
# "./ckpt_best_bs_512_fixed_alpha_sqrt_L.pt" 
# "./ckpt_best_bs_512_learnable_alpha_both_attn_mlp.pt"

DATA_DIR = "/home/cs22d010/Squad_QA/CKA-Mech-Inter-Workshop/wikitext103"
SPLIT = "val"   # "train" or "val"

#SAVE_PATH = "./wikitext103_layer_metrics_learnable_alpha_both.json"

# ---------------------------------------------------------------------
# Runtime config
# ---------------------------------------------------------------------

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device_type = "cuda" if "cuda" in str(device) else "cpu"

block_size = 512
eval_batch_size = 4        # reduce to 1 or 2 if GPU memory is tight
num_batches = 256          # total samples = num_batches * eval_batch_size * block_size tokens

dtype = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float16"
ptdtype = {
    "float32": torch.float32,
    "bfloat16": torch.bfloat16,
    "float16": torch.float16,
}[dtype]

ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(
    device_type=device_type,
    dtype=ptdtype
)

torch.manual_seed(0)
random.seed(0)

# ---------------------------------------------------------------------
# Load WikiText103 token bin
# ---------------------------------------------------------------------

bin_path = os.path.join(DATA_DIR, f"{SPLIT}.bin")
data = np.memmap(bin_path, dtype=np.uint16, mode="r")

print(f"Loaded {SPLIT}.bin with {len(data):,} tokens")

# ---------------------------------------------------------------------
# Load trained checkpoint
# ---------------------------------------------------------------------

def load_trained_model(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device)

    model_args = checkpoint["model_args"]

    # dropout should be 0 during evaluation
    model_args["dropout"] = 0.0

    gptconf = GPTConfig(**model_args)
    model = GPT(gptconf)

    state_dict = checkpoint["model"]

    # remove possible torch.compile prefix
    unwanted_prefix = "_orig_mod."
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    print(f"Loaded checkpoint: {ckpt_path}")
    print(f"Checkpoint iter_num: {checkpoint.get('iter_num', None)}")
    print(f"Best val loss: {checkpoint.get('best_val_loss', None)}")
    print(f"Model layers: {model.config.n_layer}")
    print(f"Model block size: {model.config.block_size}")
    print(f"Model vocab size: {model.config.vocab_size}")

    return model

model = load_trained_model(CKPT_PATH)

# ---------------------------------------------------------------------
# Batch sampler from WikiText103
# ---------------------------------------------------------------------

def get_random_batch(data, batch_size, block_size, device):
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([
        torch.from_numpy(data[i:i + block_size].astype(np.int64))
        for i in ix
    ])

    x = x.to(device, non_blocking=True)
    return x

Loaded val.bin with 287,645 tokens
number of parameters: 123.61M
Loaded checkpoint: ./ckpt_best_bs_512_learnable_alpha.pt
Checkpoint iter_num: 52400
Best val loss: 3.425075054168701
Model layers: 12
Model block size: 512
Model vocab size: 50304


In [4]:
# ---------------------------------------------------------------------
# Collect layer embeddings
# ---------------------------------------------------------------------

@torch.no_grad()
def collect_layer_embeddings(model, data, num_batches, batch_size, block_size):
    """
    Returns:
        layer2embs[layer] = tensor of shape [num_tokens_total, n_embd]

    Layer 0 = token + position embedding output
    Layer i = output after transformer block i
    """
    layer2chunks = {}

    for _ in tqdm(range(num_batches), desc="Collecting hidden states"):
        x = get_random_batch(data, batch_size, block_size, device)

        with ctx:
            _, _, hidden_states = model(
                x,
                targets=None,
                return_hidden_states=True
            )

        for layer_idx, h in enumerate(hidden_states):
            # h shape: [B, T, C]
            h = h.detach().float().cpu()
            h = h.reshape(-1, h.size(-1))  # [B*T, C]

            if layer_idx not in layer2chunks:
                layer2chunks[layer_idx] = []

            layer2chunks[layer_idx].append(h)

        del x, hidden_states
        torch.cuda.empty_cache()

    layer2embs = {
        layer_idx: torch.cat(chunks, dim=0)
        for layer_idx, chunks in layer2chunks.items()
    }

    return layer2embs

embs_dict = collect_layer_embeddings(
    model=model,
    data=data,
    num_batches=num_batches,
    batch_size=eval_batch_size,
    block_size=block_size
)

print("Collected layers:", sorted(embs_dict.keys()))
print("Layer 0 shape:", embs_dict[0].shape)

Collected layers: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Layer 0 shape: torch.Size([524288, 768])


In [5]:
# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------

def get_est_svd(X, Y, eps=1e-8):
    """
    Approximate Y with linear transformation Y = X A.
    X: [n_samples, dim]
    Y: [n_samples, dim]
    """
    X = X.float()
    Y = Y.float()

    U, S, Vh = torch.linalg.svd(X, full_matrices=False)

    S_inv = 1.0 / (S + eps)
    A_estimation = (Vh.T * S_inv[None, :]) @ U.T @ Y

    Y_est = X @ A_estimation
    return Y_est


def procrustes_similarity(x, y, eps=1e-8):
    """
    Procrustes-style similarity between x and y.
    """
    with torch.no_grad():
        X = x.float()
        Y = y.float()

        X = X - X.mean(dim=0, keepdim=True)
        Y = Y - Y.mean(dim=0, keepdim=True)

        X = X / (X.norm() + eps)
        Y = Y / (Y.norm() + eps)

        Y_est = get_est_svd(X, Y, eps=eps)

        y_error = (Y_est - Y).square().sum()
        sim = 1.0 - y_error

    return float(sim)

def Update_only_procustes(x, y, eps=1e-8):
    """
    Procrustes-style similarity between x and y.
    """
    with torch.no_grad():
        X = x.float()
        Y = y.float() - X 

        X = X - X.mean(dim=0, keepdim=True)
        Y = Y - Y.mean(dim=0, keepdim=True)

        X = X / (X.norm() + eps)
        Y = Y / (Y.norm() + eps)

        Y_est = get_est_svd(X, Y, eps=eps)

        y_error = (Y_est - Y).square().sum()
        sim = 1.0 - y_error

    return float(sim)


def cka_similarity(x, y, eps=1e-8, centered=False):
    """
    Linear CKA.

    x: [n_samples, dim]
    y: [n_samples, dim]
    """
    with torch.no_grad():
        X = x.float()
        Y = y.float()

        if centered:
            X = X - X.mean(dim=0, keepdim=True)
            Y = Y - Y.mean(dim=0, keepdim=True)

        XT_Y = X.T @ Y
        XT_X = X.T @ X
        YT_Y = Y.T @ Y

        numerator = (XT_Y ** 2).sum()
        denominator = torch.sqrt((XT_X ** 2).sum() * (YT_Y ** 2).sum())

        cka = numerator / (denominator + eps)

    return float(cka)


def Update_only_cka_similarity(x, y, eps=1e-8, centered=False):
    """
    Linear CKA.

    x: [n_samples, dim]
    y: [n_samples, dim]
    """
    with torch.no_grad():
        X = x.float()
        Y = y.float() - X

        if centered:
            X = X - X.mean(dim=0, keepdim=True)
            Y = Y - Y.mean(dim=0, keepdim=True)

        XT_Y = X.T @ Y
        XT_X = X.T @ X
        YT_Y = Y.T @ Y

        numerator = (XT_Y ** 2).sum()
        denominator = torch.sqrt((XT_X ** 2).sum() * (YT_Y ** 2).sum())

        cka = numerator / (denominator + eps)

    return float(cka)



def update_norm_ratio(x, y, eps=1e-8):
    """
    Computes ||y - x|| / ||x||.
    """
    with torch.no_grad():
        X = x.float()
        Y = y.float()

        delta = Y - X
        ratio = torch.linalg.vector_norm(delta) / (
            torch.linalg.vector_norm(X) + eps
        )

    return float(ratio)

In [6]:
# ---------------------------------------------------------------------
# Compute layerwise metrics
# ---------------------------------------------------------------------

all_results = {
    "procrustes_similarity": {},
    "cka_similarity": {},
    "cka_similarity_centered": {},
    "update_norm_ratio": {},
    "update_only_cka_similarity":{},
    "update_only_procustes":{}
}

model_key = os.path.basename(CKPT_PATH)

max_layer = max(embs_dict.keys())

procrustes_scores = []
cka_scores = []
cka_centered_scores = []
norm_ratios = []
update_only_cka = []
update_only_procustes = []

for layer in tqdm(range(max_layer), desc="Computing layerwise metrics"):
    x = embs_dict[layer]
    y = embs_dict[layer + 1]

    procrustes_scores.append(procrustes_similarity(x, y))
    cka_scores.append(cka_similarity(x, y, centered=False))
    cka_centered_scores.append(cka_similarity(x, y, centered=True))
    norm_ratios.append(update_norm_ratio(x, y))
    update_only_cka.append(Update_only_cka_similarity(x,y))
    update_only_procustes.append(Update_only_procustes(x,y))

all_results["procrustes_similarity"][model_key] = procrustes_scores
all_results["cka_similarity"][model_key] = cka_scores
all_results["cka_similarity_centered"][model_key] = cka_centered_scores
all_results["update_norm_ratio"][model_key] = norm_ratios
all_results["update_only_cka_similarity"][model_key] = update_only_cka
all_results["update_only_procustes"][model_key] = update_only_procustes

#with open(SAVE_PATH, "w") as f:
#    json.dump(all_results, f, indent=2)

#print(f"Saved results to {SAVE_PATH}")

Computing layerwise metrics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [09:23<00:00, 46.92s/it]


In [7]:
all_results

{'procrustes_similarity': {'ckpt_best_bs_512_learnable_alpha.pt': [0.6253829002380371,
   0.72362220287323,
   0.7439368963241577,
   0.9505054354667664,
   0.9953744411468506,
   0.9987868070602417,
   0.9996591210365295,
   0.9997937679290771,
   0.9998612999916077,
   0.9999122619628906,
   0.999943196773529,
   0.9999614357948303]},
 'cka_similarity': {'ckpt_best_bs_512_learnable_alpha.pt': [0.5282458066940308,
   0.7939327955245972,
   0.8115503787994385,
   0.9554200172424316,
   0.9747352600097656,
   0.9757797122001648,
   0.9933359026908875,
   0.9957363605499268,
   0.9971853494644165,
   0.9979426264762878,
   0.9984335899353027,
   0.9987035393714905]},
 'cka_similarity_centered': {'ckpt_best_bs_512_learnable_alpha.pt': [0.5363284945487976,
   0.7451136112213135,
   0.8054159283638,
   0.9579513669013977,
   0.9756366610527039,
   0.9764434695243835,
   0.993869960308075,
   0.9961172342300415,
   0.9974443912506104,
   0.9982185959815979,
   0.9986698031425476,
   0.998982

In [8]:
def print_learned_alphas(model):
    """
    Prints learned alpha/scale values for each transformer block.

    Works for common names:
    - block.scale
    - block.scale_1 / block.scale_2
    - block.alpha
    - block.alpha_1 / block.alpha_2
    """
    for i, block in enumerate(model.transformer.h):
        print(f"\nLayer {i}")

        found = False

        for name in ["scale", "alpha", "scale_1", "scale_2", "alpha_1", "alpha_2"]:
            if hasattr(block, name):
                value = getattr(block, name)

                if isinstance(value, torch.nn.Parameter) or torch.is_tensor(value):
                    value = value.detach().cpu().float().item()
                else:
                    value = float(value)

                print(f"  {name}: {value:.8f}")
                found = True

        if not found:
            print("  No alpha/scale attribute found")

In [9]:
print_learned_alphas(model)


Layer 0
  scale: -0.12280592

Layer 1
  scale: -0.62283558

Layer 2
  scale: 1.56988335

Layer 3
  scale: -1.88566864

Layer 4
  scale: -1.19898367

Layer 5
  scale: -0.79182971

Layer 6
  scale: 0.58240747

Layer 7
  scale: -0.50875765

Layer 8
  scale: 0.45409143

Layer 9
  scale: -0.38064441

Layer 10
  scale: -0.32245085

Layer 11
  scale: -0.27923483


In [ ]:
dataset = 'Squad_QA/CKA-Mech-Inter-Workshop/wikitext103' 
data_dir = os.path.join('/home/cs22d010', dataset)

batch_size = 16 # if gradient_accumulation_steps > 1, this is the micro-batch size  #1801350/160=11258 batches
block_size = 512


In [ ]:
def get_batch(split):
    # We recreate np.memmap every batch to avoid a memory leak, as per
    # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
    if split == 'train':
        data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint16, mode='r')
    else:
        data = np.memmap(os.path.join(data_dir, 'val.bin'), dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


In [ ]:
@torch.no_grad()
def linear_cka(X, Y, eps=1e-8):
    """
    Linear CKA between two representation matrices.

    X, Y: tensors of shape [N, D]
    """
    X = X.float()
    Y = Y.float()

    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)

    XT_Y = X.T @ Y
    XT_X = X.T @ X
    YT_Y = Y.T @ Y

    hsic = torch.linalg.matrix_norm(XT_Y, ord="fro") ** 2
    var1 = torch.linalg.matrix_norm(XT_X, ord="fro")
    var2 = torch.linalg.matrix_norm(YT_Y, ord="fro")

    return (hsic / (var1 * var2 + eps)).item()

In [ ]:
@torch.no_grad()
def compute_adjacent_cka_from_embs(embs_dict, max_samples=None, seed=0):
    """
    Computes CKA between consecutive hidden states.

    embs_dict[0] = embedding output
    embs_dict[1] = after block 0
    ...
    embs_dict[L] = after block L-1

    Returns:
        adj_cka[i] = CKA(embs_dict[i], embs_dict[i+1])
        This corresponds to the transition induced by block i.
    """
    layers = sorted(embs_dict.keys())
    n_transitions = len(layers) - 1

    adj_cka = []

    rng = torch.Generator()
    rng.manual_seed(seed)

    for i in range(n_transitions):
        X = embs_dict[i]
        Y = embs_dict[i + 1]

        assert X.shape == Y.shape, f"Shape mismatch at transition {i}: {X.shape}, {Y.shape}"

        if max_samples is not None and X.size(0) > max_samples:
            idx = torch.randperm(X.size(0), generator=rng)[:max_samples]
            X_use = X[idx]
            Y_use = Y[idx]
        else:
            X_use = X
            Y_use = Y

        cka_val = linear_cka(X_use, Y_use)
        adj_cka.append(cka_val)

    return adj_cka

In [ ]:
adj_cka = compute_adjacent_cka_from_embs(
    embs_dict,
    max_samples=50000,   # use None if memory is fine
    seed=0
)

for i, v in enumerate(adj_cka):
    print(f"Block {i:02d}: CKA(h_{i}, h_{i+1}) = {v:.4f}")

In [ ]:
def find_high_cka_runs(
    adj_cka,
    threshold=0.95,
    ignore_first=True,
    ignore_last=True,
    min_run_length=1,
):
    """
    Finds consecutive block indices whose adjacent CKA is above threshold.

    adj_cka[i] corresponds to block i.
    """
    L = len(adj_cka)

    start = 1 if ignore_first else 0
    end = L - 1 if ignore_last else L

    selected = [
        i for i in range(start, end)
        if adj_cka[i] >= threshold
    ]

    runs = []
    current = []

    for i in selected:
        if len(current) == 0 or i == current[-1] + 1:
            current.append(i)
        else:
            if len(current) >= min_run_length:
                runs.append(current)
            current = [i]

    if len(current) >= min_run_length:
        runs.append(current)

    return runs

In [ ]:
runs = find_high_cka_runs(
    adj_cka,
    threshold=0.97,
    ignore_first=True,
    ignore_last=True,
    min_run_length=1,
)

print("High-CKA runs:", runs)

In [ ]:
import torch.nn.functional as F
from contextlib import nullcontext

@torch.no_grad()
def forward_with_multi_layer_ablation(model, idx, targets, ablate_layers=None):
    """
    Forward pass where multiple transformer blocks are skipped.

    ablate_layers:
        list/set of block indices to remove.
        Example: [4, 5, 6]
    """
    model.eval()

    if ablate_layers is None:
        ablate_layers = set()
    else:
        ablate_layers = set(ablate_layers)

    device = idx.device
    b, t = idx.size()

    assert t <= model.config.block_size

    pos = torch.arange(0, t, dtype=torch.long, device=device)

    tok_emb = model.transformer.wte(idx)
    pos_emb = model.transformer.wpe(pos)
    x = model.transformer.drop(tok_emb + pos_emb)

    for layer_idx, block in enumerate(model.transformer.h):
        if layer_idx in ablate_layers:
            # Skip this block completely.
            # x_{layer+1} = x_layer
            pass
        else:
            x = block(x)

    logits = model.lm_head(x)

    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        targets.view(-1),
        ignore_index=-1,
    )

    return logits, loss

In [ ]:
@torch.no_grad()
def estimate_group_ablation_losses(
    model,
    get_batch,
    layer_groups,
    split="val",
    eval_iters=100,
    ctx=None,
):
    """
    Computes original and group-ablated loss.

    layer_groups:
        list of lists, e.g.
        [[4, 5, 6], [8, 9, 10]]
    """
    if ctx is None:
        ctx = nullcontext()

    model.eval()

    original_losses = torch.zeros(eval_iters)
    group_losses = {
        tuple(group): torch.zeros(eval_iters)
        for group in layer_groups
    }

    for k in range(eval_iters):
        X, Y = get_batch(split)

        with ctx:
            _, original_loss = forward_with_multi_layer_ablation(
                model,
                X,
                Y,
                ablate_layers=None,
            )

        original_losses[k] = original_loss.item()

        for group in layer_groups:
            group_key = tuple(group)

            with ctx:
                _, ablated_loss = forward_with_multi_layer_ablation(
                    model,
                    X,
                    Y,
                    ablate_layers=group,
                )

            group_losses[group_key][k] = ablated_loss.item()

        if (k + 1) % 10 == 0:
            print(f"Finished {k + 1}/{eval_iters} batches")

    original_mean = original_losses.mean().item()

    results = []

    for group in layer_groups:
        group_key = tuple(group)
        ablated_mean = group_losses[group_key].mean().item()

        results.append({
            "layers_removed": list(group),
            "num_layers_removed": len(group),
            "original_loss": original_mean,
            "ablated_loss": ablated_mean,
            "delta_loss": ablated_mean - original_mean,
            "original_ppl": float(np.exp(original_mean)),
            "ablated_ppl": float(np.exp(ablated_mean)),
        })

    return results

In [ ]:
group_results = estimate_group_ablation_losses(
    model=model,
    get_batch=get_batch,
    layer_groups=runs,
    split="val",
    eval_iters=100,
    ctx=ctx,
)

for r in group_results:
    print(
        f"Removed layers {r['layers_removed']} | "
        f"orig loss {r['original_loss']:.4f} | "
        f"ablated loss {r['ablated_loss']:.4f} | "
        f"delta {r['delta_loss']:.4f} | "
        f"orig ppl {r['original_ppl']:.2f} | "
        f"ablated ppl {r['ablated_ppl']:.2f}"
    )